# Classifying 200 Bird Species from CUB_200_2011 Dataset with Multilayer Perceptron

### About

This notebook uses the Sklearn library to implement a Multilayer Perceptron to gauge its performance on the fine-grained CUB_200_2011 dataset.

The notebook relies on previously completed data preparation in `dataPrep.ipynb`.

### Contents

This notebook is organised as follows.

1. Setup and Installation
2. Getting to Know Our Data
3. MLP Model Training
4. Evaluation and Results
5. Tuning Hyperparameters to Curb Overfitting

NOTE: Section 1 and 2 have been copied from `notebooks/logistic_regression.ipynb` due to similar requirements for those sections.

### References

1. [Sklearn Neural Networks Supervised](https://scikit-learn.org/stable/modules/neural_networks_supervised.html)
2. [Sklean MLP Classifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

## Section 1: Installation and Setup

In [1]:
import numpy as np
from sklearn.neural_network import MLPClassifier
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load data
data_path="../data/cub_features_A.csv"
df = pd.read_csv(data_path)

## Section 2: Getting to Know our Data

The next part explores different features of the data

In [3]:
# View the shape of our data
df.shape

(11788, 317)

In [4]:
# View an example of the records in our data
df.head()

,image_id,has_bill_shape::curved_(up_or_down),has_bill_shape::dagger,has_bill_shape::hooked,has_bill_shape::needle,has_bill_shape::hooked_seabird,has_bill_shape::spatulate,has_bill_shape::all-purpose,has_bill_shape::cone,has_bill_shape::specialized,...,has_crown_color::red,has_crown_color::buff,has_wing_pattern::solid,has_wing_pattern::spotted,has_wing_pattern::striped,has_wing_pattern::multi-colored,class_id,class_name,is_train,is_train_80
0,1,0,0,0,0,1,0,0,0,0,...,0,1,0,0,0,0,1,001.Black_footed_Albatross,0,1
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,001.Black_footed_Albatross,1,1
2,3,0,0,0,0,1,0,0,0,0,...,0,1,0,0,1,0,1,001.Black_footed_Albatross,0,1
3,4,0,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,1,001.Black_footed_Albatross,1,1
4,5,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,001.Black_footed_Albatross,1,1


In [5]:
col_names = df.columns
col_names

Index(['image_id', 'has_bill_shape::curved_(up_or_down)',
       'has_bill_shape::dagger', 'has_bill_shape::hooked',
       'has_bill_shape::needle', 'has_bill_shape::hooked_seabird',
       'has_bill_shape::spatulate', 'has_bill_shape::all-purpose',
       'has_bill_shape::cone', 'has_bill_shape::specialized',
       ...
       'has_crown_color::red', 'has_crown_color::buff',
       'has_wing_pattern::solid', 'has_wing_pattern::spotted',
       'has_wing_pattern::striped', 'has_wing_pattern::multi-colored',
       'class_id', 'class_name', 'is_train', 'is_train_80'],
      dtype='object', length=317)

In [6]:
# View summary of dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11788 entries, 0 to 11787
Columns: 317 entries, image_id to is_train_80
dtypes: int64(316), object(1)
memory usage: 28.5+ MB


In [7]:
# Ensure there is no missing data
df.isnull().sum()

image_id                               0
has_bill_shape::curved_(up_or_down)    0
has_bill_shape::dagger                 0
has_bill_shape::hooked                 0
has_bill_shape::needle                 0
                                      ..
has_wing_pattern::multi-colored        0
class_id                               0
class_name                             0
is_train                               0
is_train_80                            0
Length: 317, dtype: int64

## Section 3: MLP Model Training

### Section 3.1: Separate training and test data with attributes and target class

In [8]:
SPLIT = "is_train" # Use recommended 50-50 split
META_COLS = ["class_id", "image_id", "class_name", "is_train"]

# Split training and testing data
train, test = df[df[SPLIT] == 1], df[df[SPLIT] == 0]

# Get all feature columns
feature_cols = [col for col in df.columns if col not in META_COLS]
X_train, y_train = train[feature_cols], train.class_id
X_test, y_test = test[feature_cols], test.class_id

print("X_train.shape: ", X_train.shape)
print("X_test.shape: ", X_test.shape)
print("y_train.shape: ", y_train.shape)
print("y_test.shape: ", y_test.shape)

X_train.shape:  (5994, 313)
X_test.shape:  (5794, 313)
y_train.shape:  (5994,)
y_test.shape:  (5794,)


### Section 3.2: Begin Training the Model

In [9]:
# Initialise and train model
mlp = MLPClassifier(activation='relu', hidden_layer_sizes=(512, 384, 256), max_iter=1000, tol=1e-4)

In [10]:
mlp.fit(X_train, y_train)

MLPClassifier(hidden_layer_sizes=(512, 384, 256), max_iter=1000)

## Section 4: Results and Evaluation

In [ ]:
y_preds_train = mlp.predict(X_train)
y_preds_test = mlp.predict(X_test)
from sklearn.metrics import accuracy_score, f1_score

print(f"Model accuracy (training set): {accuracy_score(y_train, y_preds_train) * 100:.4f}%")
print(f"Model accuracy (test set): {accuracy_score(y_test, y_preds_test) * 100:.4f}%")
print(f"Model f1-score (training set): {f1_score(y_train, y_preds_train, average="macro"):.4f}")
print(f"Model f1-score (test set): {f1_score(y_test, y_preds_test, average="macro"):.4f}")

Model accuracy (training set): 99.7998%
Model accuracy (test set): 45.6679%
Model f1-score (training set): 0.9980
Model accuracy (test set): 0.4579


The model is overfitting. This is clear from the stark difference between the training and test accuracy scores. Claude recommends retrying with `early_stopping` enabled when consulted on how to rectify overfitting. Below are the results.

```python
early_stopping=False
```

| Metric | Result |
|--------|--------|
| Training Accuracy | 99.9% |
| Test Accuracy | 45.4% |

```python
early_stopping=True
```

| Metric | Result |
|--------|--------|
| Training Accuracy | 67.63% |
| Test Accuracy | 42.9% |

Conclusion: When `early_stopping` is, the model is prevented from overfitting. Let us try to fix that by increasing the alpha strength.


In the next run, to encourage better generalisation, we make the following changes:
1. Increase alpha which is the strength of the regularisation term
2. Reduce the number of hidden layers
3. Enable early_stopping to prevent overfitting
4. Use a smaller batch size

## Tuning Hyperparameters to Curb Overfitting

In [ ]:
# Retrain model
mlp = MLPClassifier(
    activation='relu',
    hidden_layer_sizes=(384, 256),
    batch_size=32,
    max_iter=1000,
    tol=1e-8,
    alpha=1e-2,
    early_stopping=True
).fit(X_train, y_train)

y_preds_train = mlp.predict(X_train)
y_preds_test = mlp.predict(X_test)
from sklearn.metrics import accuracy_score, f1_score

print(f"Model accuracy (training set): {accuracy_score(y_train, y_preds_train) * 100:.4f}%")
print(f"Model accuracy (test set): {accuracy_score(y_test, y_preds_test) * 100:.4f}%")
print(f"Model f1-score (training set): {f1_score(y_train, y_preds_train, average="macro"):.4f}")
print(f"Model f1-score (test set): {f1_score(y_test, y_preds_test, average="macro"):.4f}")

Model accuracy (training set): 93.6770%
Model accuracy (test set): 45.1847%
Model f1-score (training set): 0.9368
Model accuracy (test set): 0.4494


A better accuracy and f1 score is achieved with lower overfitting. Overall, the accuracy may still be less than impressive. This is perhaps due to the fine-grained nature of this dataset which makes it exhibit such complex patterns. In the next notebooks, we will explore more complex neural networks that use the bird's image data for classification.

In [ ]:
from sklearn.model_selection import GridSearchCV
from time import time

PARAM_GRID = [
    {
        "hidden_layer_sizes": [(384, 256), (512, 384), (512, 256)],
        "alpha": [1e-2, 1e-3],
        "max_iter": [500],
        "tol": [1e-4],
        "early_stopping": [True]
    }
]

t0 = time()
search = GridSearchCV(
    MLPClassifier(),
    param_grid=PARAM_GRID,
    scoring="f1_macro",
    refit="f1_macro",
    cv=3,
    # n_jobs=-1,
    return_train_score=True,
    verbose=4
).fit(X_train, y_train)

delta = round(time() - t0)

print(f'{len(search.cv_results_["params"])} combinations x 3 folds in {delta}s')

print("Best Params\n", search.best_params_)
print("Best Score\n", search.best_score_)

model = search.best_estimator_
y_preds_train = model.predict(X_train)
y_preds_test = model.predict(X_test)

print(f"Model accuracy (training set): {accuracy_score(y_train, y_preds_train) * 100:.4f}%")
print(f"Model accuracy (test set): {accuracy_score(y_test, y_preds_test) * 100:.4f}%")
print(f"Model f1-score (training set): {f1_score(y_train, y_preds_train, average="macro"):.4f}")
print(f"Model f1-score (test set): {f1_score(y_test, y_preds_test, average="macro"):.4f}")

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV 1/3] END alpha=0.01, early_stopping=True, hidden_layer_sizes=(384, 256), max_iter=500, tol=0.0001;, score=(train=0.932, test=0.431) total time=  10.8s
[CV 2/3] END alpha=0.01, early_stopping=True, hidden_layer_sizes=(384, 256), max_iter=500, tol=0.0001;, score=(train=0.788, test=0.411) total time=   5.9s
[CV 3/3] END alpha=0.01, early_stopping=True, hidden_layer_sizes=(384, 256), max_iter=500, tol=0.0001;, score=(train=0.847, test=0.418) total time=   6.3s
[CV 1/3] END alpha=0.01, early_stopping=True, hidden_layer_sizes=(512, 384), max_iter=500, tol=0.0001;, score=(train=0.856, test=0.421) total time=  11.7s
[CV 2/3] END alpha=0.01, early_stopping=True, hidden_layer_sizes=(512, 384), max_iter=500, tol=0.0001;, score=(train=0.920, test=0.439) total time=  13.9s
[CV 3/3] END alpha=0.01, early_stopping=True, hidden_layer_sizes=(512, 384), max_iter=500, tol=0.0001;, score=(train=0.930, test=0.426) total time=  15.4s
[CV 1/3] E

The 3-fold Grid Search reveals the best combination of hyperparameters:

```py
{
    'alpha': 0.001,
    'early_stopping': True,
    'hidden_layer_sizes': (512, 384),
    'max_iter': 500,
    'tol': 0.0001
}
```

which result in the best test set performance so far:

| Metric | Result |
|--------|--------|
| Test Accuracy | 46.82% |
| Test F1-Score | 0.4710 |
